# 00 — Setup: ตรวจสภาพแวดล้อมของโปรเจกต์

> **ขั้น 0 ของ workflow** · ทุกคนในกลุ่มรัน notebook นี้หลังติดตั้ง `.venv` ครั้งแรก
> ถ้าทุก cell ผ่าน = เครื่องของเรามีไลบรารีเวอร์ชันเดียวกัน และหาไฟล์ข้อมูลเจอ → ผลของ notebook อื่นจะ reproduce ได้ตรงกันทั้ง 3 เครื่อง

วิธีติดตั้ง (ทำครั้งเดียว, Windows):
```
py -3.13 -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
```
แล้วเลือก kernel ของ `.venv` ใน Jupyter/VS Code

## 0.1 เวอร์ชัน Python และไลบรารี

**โค้ดทำอะไร:** อ่านเวอร์ชันที่ pin ไว้ใน `requirements.txt` (บรรทัดรูปแบบ `ชื่อ==เวอร์ชัน`) แล้วเทียบกับเวอร์ชันที่ติดตั้งจริงในเครื่องผ่าน `importlib.metadata`
- ใช้ `importlib.metadata.version()` แทนการ `import` ทีละตัว เพราะชื่อแพ็กเกจบน pip กับชื่อโมดูลบางตัวไม่ตรงกัน (เช่น `scikit-learn` vs `sklearn`)
- ถ้าเวอร์ชันไม่ตรง ผลลัพธ์ตัวเลข (เช่นคะแนน CV) อาจต่างกันเล็กน้อยระหว่างเครื่องสมาชิก จึงต้องเช็คตั้งแต่ต้น

In [1]:
# ขั้น 0 — ตรวจ environment: ทุกคนในกลุ่มต้องได้ผลเหมือนกัน จึงต้องใช้ไลบรารีเวอร์ชันเดียวกับ requirements.txt
import sys
import importlib.metadata as md_
from pathlib import Path

# รัน notebook จากโฟลเดอร์ notebooks/ หรือจาก root ก็ได้ → หา root ของ repo แล้วใส่ใน sys.path เพื่อ import config
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

print("Python", sys.version.split()[0])
rows = []
for line in (ROOT / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()          # ตัด comment ท้ายบรรทัดใน requirements.txt
    if "==" not in line:
        continue
    pkg, pinned = line.split("==")
    try:
        installed = md_.version(pkg)           # เวอร์ชันที่ติดตั้งจริงใน .venv
    except md_.PackageNotFoundError:
        installed = "ไม่ได้ติดตั้ง"
    rows.append((pkg, pinned, installed, "OK" if installed == pinned else "ไม่ตรง"))

for r in rows:
    print(f"{r[0]:<14} pin={r[1]:<8} installed={r[2]:<10} {r[3]}")
# หยุดทันทีถ้าเวอร์ชันไม่ตรง — ผลลัพธ์ (เช่น MAE) อาจต่างจากที่รายงานไว้
assert all(r[3] == "OK" for r in rows), "มีไลบรารีเวอร์ชันไม่ตรงกับ requirements.txt"

Python 3.13.11
numpy          pin=2.5.3    installed=2.5.3      OK
pandas         pin=2.3.3    installed=2.3.3      OK
scipy          pin=1.18.1   installed=1.18.1     OK
scikit-learn   pin=1.9.1    installed=1.9.1      OK
matplotlib     pin=3.11.2   installed=3.11.2     OK
seaborn        pin=0.13.2   installed=0.13.2     OK
joblib         pin=1.6.0    installed=1.6.0      OK
ucimlrepo      pin=0.0.7    installed=0.0.7      OK
notebook       pin=7.6.3    installed=7.6.3      OK
ipykernel      pin=7.3.0    installed=7.3.0      OK
nbclient       pin=0.11.0   installed=0.11.0     OK


**ผลลัพธ์:** ไลบรารีทุกตัวขึ้น `OK` = เวอร์ชันตรงกับที่ pin ไว้ทุกตัว (`assert` ผ่าน ไม่มี error)
- สิ่งที่ต้องจำ: ใช้ **pandas 2.3.3** ไม่ใช่ 3.x เพราะบนเครื่องนี้ Windows Smart App Control บล็อก DLL ของ pandas 3 (บันทึกไว้ใน `requirements.txt`)
- Python เป็น **3.13** ตรงกับที่ `requirements.txt` ระบุ (cell นี้ตรวจเฉพาะไลบรารี ไม่ได้ assert เวอร์ชัน Python) · ตรวจแล้วว่า `01_eda` และ `02_modeling` ได้ตัวเลขเหมือนกันทุก cell ทั้งบน 3.13 และ 3.14 (ต่างกันแค่เวลา fit/latency)
- ถ้าเครื่องสมาชิกคนไหนขึ้น `ไม่ตรง` ให้รัน `pip install -r requirements.txt` ใหม่ใน `.venv`

## 0.2 ค่าคงที่ใน `config.py` และไฟล์ข้อมูล

**โค้ดทำอะไร:** import `config.py` (ไฟล์กลางที่ทุก notebook ใช้ร่วมกัน) แล้วพิมพ์ค่าที่สำคัญ + ตรวจว่าไฟล์ CSV ดิบอยู่ในตำแหน่งที่ `config.RAW_CSV` ชี้ไว้จริง
- `RANDOM_STATE` ใช้ทุกจุดที่มีการสุ่ม → รันซ้ำได้ผลเดิม
- `TASK` บอกชนิดงาน — ตอนนี้เป็น `regression` (ทำนายยอดเช่า คัน/ชม.)

In [2]:
# ค่าคงที่ทั้งหมดอยู่ที่ config.py ที่เดียว → ทุก notebook ใช้ seed, สัดส่วน test และ path ชุดเดียวกัน
import config

print("RANDOM_STATE =", config.RANDOM_STATE)
print("TEST_FRAC    =", config.TEST_FRAC, "(สัดส่วนจำนวน 'วัน' ของ test)")
print("TASK         =", config.TASK)
print("RAW_CSV      =", config.RAW_CSV.relative_to(ROOT))
# ตรวจว่ามีไฟล์ข้อมูลดิบก่อนเริ่ม 01_eda
assert config.RAW_CSV.exists(), "ไม่เจอไฟล์ข้อมูล — แตก data/raw/seoul+bike+sharing+demand.zip ก่อน"
print(f"ขนาดไฟล์     = {config.RAW_CSV.stat().st_size:,} bytes")
for d in [config.SPLIT_DIR, config.FIGURES_DIR, config.REPORTS_DIR]:
    print(f"{str(d.relative_to(ROOT)):<14} exists={d.exists()}")

RANDOM_STATE = 42
TEST_FRAC    = 0.2 (สัดส่วนจำนวน 'วัน' ของ test)
TASK         = regression
RAW_CSV      = data\raw\seoul+bike+sharing+demand\SeoulBikeData.csv
ขนาดไฟล์     = 604,166 bytes
data\splits    exists=True
figures        exists=True
reports        exists=True


**ผลลัพธ์:** หาไฟล์ `SeoulBikeData.csv` เจอ (≈604 KB ตรงกับขนาดใน zip ของ UCI) และโฟลเดอร์ `data/splits`, `figures`, `reports` มีอยู่ครบ
- `TASK = regression` → ทำนายจำนวนจักรยานที่ถูกเช่า (คัน/ชม.)
- ไฟล์ดิบอยู่ในโฟลเดอร์ย่อย `data/raw/seoul+bike+sharing+demand/` (ตามที่แตกจาก zip) — แก้ path ใน `config.py` ให้ตรงแล้ว

## 0.3 กติกาการทำงานของกลุ่ม

**แบ่งงาน**
| คน | รับผิดชอบหลัก | notebook |
|---|---|---|
| A | ขั้น 2–4: โหลด/ตรวจข้อมูล, split, EDA | `01_eda.ipynb` |
| B | ขั้น 5–8: feature engineering, Pipeline, feature selection, PCA | `02_modeling.ipynb` (ส่วนต้น) |
| C | ขั้น 9–12: baseline, เทียบโมเดล, tuning, ตาราง CLO4 | `02_modeling.ipynb` (ส่วนท้าย) |

**กติกา git**
- **1 ขั้น = 1 branch** ตั้งชื่อ `step/<เลขขั้น>-<ชื่อสั้น>` เช่น `step/3-split` · เปิด PR เข้า `main` แล้วให้อีก 2 คนรีวิวก่อน merge
- ห้ามแก้ notebook เดียวกันพร้อมกันคนละ branch — JSON ของ notebook merge ยากมาก ถ้าต้องทำคู่ขนานให้แยก notebook
- ก่อน commit ให้ **Restart & Run All** ทุกครั้ง เพื่อให้ output ที่ commit ตรงกับโค้ดจริง (เลข execution เรียง 1, 2, 3, …)
- **ไม่ติดตั้ง `nbstripout`** โดยตั้งใจ: เราต้องการเก็บ output (ตาราง/กราฟ) ไว้ใน commit เป็นหลักฐานประกอบคำอธิบายใต้แต่ละ cell · ลด conflict ด้วยกติกา "คนละ notebook / Restart & Run All" แทน

**สิ่งที่ต้องทำเอง (นอก notebook)**
- [ ] push repo ขึ้น GitHub และเพิ่ม collaborator ครบ 3 คน
- [x] ตัดสินใจชนิดงาน: ใช้ **regression** (dataset มี target เป็นจำนวนนับต่อเนื่อง เหมาะกว่า classification) — merge เข้า `main` แล้วใน PR #1
- [ ] กรอกรหัสนิสิต/ชื่อใน `README.md`